In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

from datetime import datetime 
import xgboost as xgb

import os

In [4]:
data = ['transaction_id', 'is_fraud', 'created_at', 'is_subscription', 'transaction_type',
        'currency_amount', 'currency_id', 'amount_scaled', 'merchant_customer_id',
        'merchant_country', 'ip_address', 'platform', 'merchant_id', 'merchant_shop_id',
        'merchant_shop_name', 'is_secured', 'ip_country', 'payment_type', 'card_id', 
        'bank', 'cardcountry', 'bin', 'card_holder_first_name', 'card_holder_last_name']

'''created_at
is_subscription
transaction_type
currency_amount
currency_id
amount_scaled
merchant_customer_id
merchant_customer_email
#merchant_customer_phone
#merchant_customer_first_name
#merchant_customer_last_name
merchant_country
#merchant_city
merchant_language
ip_address
platform
merchant_id
merchant_shop_id
merchant_shop_name
is_secured
#order_number
ip_country
#is_verified
payment_type
#traffic_source
#transaction_source
user_agent
#browser
#browser_version
#operating_system
#operating_system_version
#device
card_id
bank
cardbrand
cardcountry
cardtype
bin
card_exp_relative
card_holder_first_name
card_holder_last_name'''

'created_at\nis_subscription\ntransaction_type\ncurrency_amount\ncurrency_id\namount_scaled\nmerchant_customer_id\nmerchant_customer_email\n#merchant_customer_phone\n#merchant_customer_first_name\n#merchant_customer_last_name\nmerchant_country\n#merchant_city\nmerchant_language\nip_address\nplatform\nmerchant_id\nmerchant_shop_id\nmerchant_shop_name\nis_secured\n#order_number\nip_country\n#is_verified\npayment_type\n#traffic_source\n#transaction_source\nuser_agent\n#browser\n#browser_version\n#operating_system\n#operating_system_version\n#device\ncard_id\nbank\ncardbrand\ncardcountry\ncardtype\nbin\ncard_exp_relative\ncard_holder_first_name\ncard_holder_last_name'

In [ ]:
train = pd.read_csv('/kaggle/input/int20h_test_2025/train.csv', usecols=data)

train.head()

In [ ]:
!python --version

In [3]:
train_path = '/kaggle/input/int20h_test_2025/train.csv' 

In [4]:
# Generate indices to skip (keeping 75% of data for example)
total_rows = sum(1 for _ in open(train_path)) - 1  # -1 for header
indices_to_skip = set(np.random.choice(
    range(1, total_rows + 1),  # Skip from row 1 (after header)
    size=int(total_rows * 0.75),  # Skip 50% of rows
    replace=False
))

# Read CSV skipping the selected rows
df = pd.read_csv(train_path, 
                 skiprows=lambda x: x in indices_to_skip)

<ipython-input-4-4b0184a5a76e>:10: DtypeWarning: Columns (24,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(train_path,


In [5]:
df.head()

,transaction_id,is_fraud,created_at,is_subscription,transaction_type,currency_amount,currency_id,amount_scaled,merchant_customer_id,merchant_customer_email,...,device,card_id,bank,cardbrand,cardcountry,cardtype,bin,card_exp_relative,card_holder_first_name,card_holder_last_name
0,8147581512265419576,0,2024-01-28 06:31:26.253682787,True,first,1078.65,4,526,2ae2b02e168b5614f46a2b94715562012cf01f405b4da2...,53514d7fdcf50004badc5914c5faaad5311e94b9596c90...,...,NaN,1137f5c95cbd1e9c6513b475f0cc4ebf1df27afa23a6b3...,Suncorp-Metway Limited,VISA,AUS,DEBIT,6e23dbda72e8f0b5236187e648159b5cca78fb829b56f7...,43.0,NaN,NaN
1,13648480354949313259,0,2024-01-24 10:50:45.253682787,True,first,3508.65,4,1712,598fb6fd155539f799381de181b70e797fb8d95fb8b195...,8a6e8551c0910c7fa2708245b1bcecd17ea8c3fec97795...,...,NaN,1d67a084f600525977e39f6ed823468acef0b55fb0774f...,COMMONWEALTH BANK OF AUSTRALIA,MASTERCARD,AUS,DEBIT,57629399358b0beb64e63a8635251bbb748b65a537cab6...,25.0,NaN,NaN
2,9831993196115954176,0,2024-01-25 01:29:09.253682787,True,first,201.15,4,98,f71da12aae40319b06463aaea0743bb837d6ae57daad9b...,f55b3a153c12df7a7660514ea201f795dd29aca085ce13...,...,NaN,a3a725b5db4a07666b58aba4450068d5474c6f0cd79f1f...,COMMONWEALTH BANK OF AUSTRALIA,MASTERCARD,AUS,CREDIT,2970ea11ed79de67d4571d066232919d75cdcde8b6f76d...,35.0,NaN,NaN
3,18014536142395549919,0,2024-01-25 04:06:50.253682787,True,first,201.15,4,98,b250c004244f8e9baa8685b4703be3ac557398b19c2377...,f50d888713313fcf6a42286800205e52acfab62ad54a6c...,...,NaN,07d3ea4d7db7fd8c76538475859b38595b9a2cc6ee6066...,COMMONWEALTH BANK OF AUSTRALIA,MASTERCARD,AUS,DEBIT,ead17e1bf2b82a0a6a2076ed299488f21c09f77fa1372f...,32.0,NaN,NaN
4,2755652965399468914,0,2024-01-22 04:24:46.253682787,True,first,201.15,4,99,f02bfe263a64e5fc3572a643acdf2ec7bcae48c5d9b0d5...,73782a86ebef91d85f83d59e649ae09f1548eca2a35ade...,...,NaN,0cbd5eba3878bc0e789469d4d94642aba60f38fc73ccc8...,COMMONWEALTH BANK OF AUSTRALIA,MASTERCARD,AUS,DEBIT,ead17e1bf2b82a0a6a2076ed299488f21c09f77fa1372f...,32.0,NaN,NaN


In [6]:
df.isnull().sum()

transaction_id                        0
is_fraud                              0
created_at                            0
is_subscription                       0
transaction_type                      0
currency_amount                       0
currency_id                           0
amount_scaled                         0
merchant_customer_id            1320547
merchant_customer_email               3
merchant_customer_phone         8788260
merchant_customer_first_name    7978219
merchant_customer_last_name     8310152
merchant_country                      1
merchant_city                   8840146
merchant_language               2104469
ip_address                            1
platform                              0
merchant_id                           0
merchant_shop_id                      0
merchant_shop_name                    0
is_secured                            0
order_number                    8704513
ip_country                        34768
is_verified                     9069661


In [7]:
df['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.960899
1    0.039101
Name: proportion, dtype: float64

In [9]:
df.dropna(inplace=True)

In [10]:
df['is_fraud'].value_counts(normalize=True)

Series([], Name: proportion, dtype: float64)